## cross validation train val no test 

In [1]:
import codecs, re, torch, gc, json
from torch import device
from transformers import DistilBertForSequenceClassification
from torch.nn import CrossEntropyLoss
from torch.utils.data import WeightedRandomSampler
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim

from sklearn.model_selection import StratifiedKFold
import numpy as np
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score, precision_score, recall_score
import os


/home/alant/repos/tal-projet/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# loading functions
def load_movies(path2data): # 1 classe par répertoire
    alltxts = [] # init vide
    labs = []
    cpt = 0
    for cl in os.listdir(path2data): # parcours des fichiers d'un répertoire
        for f in os.listdir(path2data+cl):
            txt = open(path2data+cl+'/'+f).read()
            alltxts.append(txt)
            labs.append(cpt)
        cpt+=1 # chg répertoire = cht classe

    return alltxts,labs

# load dataset train
path = "./Dataset/movies1000/"
alltxt,alllabs = load_movies(path)
alltxt, alllabs = np.array(alltxt), np.array(alllabs)

# loading the tokenized items
data = torch.load("train_tokens_dist.pt")
input_ids = data["input_ids"]
attention_masks = data["attention_masks"]

data = torch.load("test_tokens_dist.pt")
input_ids_test = data["input_ids"]
attention_masks_test = data["attention_masks"]


In [6]:
# train version
input_ids_train = torch.cat([input_ids, input_ids_test], dim=0)
attention_masks_train = torch.cat([attention_masks, attention_masks_test], dim=0)

# Convert labels from -1/1 → 0/1 (BCE/CE convention)
alllabs = np.where(alllabs == -1, 0, alllabs)  # 0-1 labels
y_train_tens = torch.tensor(alllabs, dtype=torch.long)

# pre emptying cache
gc.collect()
torch.cuda.empty_cache()

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", dev)


device: cuda


In [7]:

# ── Cross-Validation ──────────────────────────────────────────────────────────
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

cv_results = {}

fold_idx = 0
for train_idx, val_idx in skf.split(input_ids_train, alllabs):
    fold_idx += 1
    print(f"\n{'='*80}")
    print(f"FOLD {fold_idx}/{n_splits}")
    print(f"{'='*80}")
    
    # Split data for this fold
    X_train_fold = input_ids_train[train_idx]
    mask_train_fold = attention_masks_train[train_idx]
    y_train_fold = y_train_tens[train_idx]
    
    X_val_fold = input_ids_train[val_idx]
    mask_val_fold = attention_masks_train[val_idx]
    y_val_fold = y_train_tens[val_idx]
    
    # Compute weights for weighted sampler
    _, counts = np.unique(alllabs[train_idx], return_counts=True)
    weights = 1/counts
    sample_weights = [weights[label] for label in alllabs[train_idx]]
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(alllabs[train_idx]), replacement=True)
    
    # Create dataloaders
    dataset_train = TensorDataset(X_train_fold, mask_train_fold, y_train_fold)
    dataset_val = TensorDataset(X_val_fold, mask_val_fold, y_val_fold)
    
    train_dataloader = DataLoader(dataset_train, batch_size=16, sampler=sampler)
    val_dataloader = DataLoader(dataset_val, batch_size=16, shuffle=False)
    
    # Load fresh model for this fold
    from transformers import DistilBertConfig
    
    config = DistilBertConfig.from_pretrained('distilbert-base-cased',
        dropout=0.2,
        attention_dropout=0.2,
        seq_classif_dropout=0.4
    )
    
    model = DistilBertForSequenceClassification.from_pretrained(
        "bert_dist_cache/models--distilbert-base-cased/snapshots/6ea81172465e8b0ad3fddeed32b986cdcdcffcf0", 
        config=config
    )
    
    # Freeze all BERT layers, train only the classifier head
    for param in model.distilbert.parameters():
        param.requires_grad = False
    
    # Unfreeze only the last N encoder layers
    for layer in model.distilbert.transformer.layer[-4:]:  # last 4 layers
        for param in layer.parameters():
            param.requires_grad = True
    
    model.to(dev)
    
    class_weights = torch.tensor(weights, dtype=torch.float).to(dev)
    loss_fn = CrossEntropyLoss(weight=class_weights)
    
    # Setup optimizer
    optimizer = optim.Adam(model.parameters(), lr=2e-5)
    epochs = 5
    
    fold_epochs_evolution = {}
    
    # ── Training loop ────────────────────────────────────────────────────────
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        all_train_preds = []
        all_train_labels = []
        all_train_logits = []
        
        for i, batch in enumerate(train_dataloader):
            b_input_ids, b_attention_mask, b_labels = [t.to(dev) for t in batch]
            model.zero_grad()
            
            outputs = model(
                input_ids=b_input_ids,
                attention_mask=b_attention_mask,
            )
            logits = outputs.logits
            loss = loss_fn(logits, b_labels)
            
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            
            preds = torch.argmax(logits, dim=1)
            all_train_preds.extend(preds.cpu().numpy())
            all_train_labels.extend(b_labels.cpu().numpy())
            all_train_logits.append(logits.detach().cpu())
            
            del b_input_ids, b_attention_mask, b_labels
            del outputs, loss, logits, preds
            torch.cuda.empty_cache()
            gc.collect()
        
        avg_train_loss = total_loss / len(train_dataloader)
        
        # Training metrics
        train_f1        = f1_score(all_train_labels, all_train_preds)
        train_precision = precision_score(all_train_labels, all_train_preds)
        train_recall    = recall_score(all_train_labels, all_train_preds)
        
        all_logits_tensor = torch.cat(all_train_logits, dim=0)
        train_probs = torch.softmax(all_logits_tensor, dim=1)[:, 1].numpy()
        train_auc = roc_auc_score(all_train_labels, train_probs)
        train_ap  = average_precision_score(all_train_labels, train_probs)
        
        # ── Validation metrics ───────────────────────────────────────────────
        model.eval()
        all_val_preds = []
        all_val_labels = []
        all_val_logits = []
        
        with torch.no_grad():
            for batch in val_dataloader:
                b_input_ids, b_attention_mask, b_labels = [t.to(dev) for t in batch]
                
                outputs = model(input_ids=b_input_ids, attention_mask=b_attention_mask)
                logits = outputs.logits
                
                preds = torch.argmax(logits, dim=1)
                all_val_preds.extend(preds.cpu().numpy())
                all_val_labels.extend(b_labels.cpu().numpy())
                all_val_logits.append(logits.detach().cpu())
                
                del b_input_ids, b_attention_mask, b_labels
                del outputs, logits, preds
                torch.cuda.empty_cache()
                gc.collect()
        
        # Validation metrics
        val_f1        = f1_score(all_val_labels, all_val_preds)
        val_precision = precision_score(all_val_labels, all_val_preds)
        val_recall    = recall_score(all_val_labels, all_val_preds)
        
        all_val_logits_tensor = torch.cat(all_val_logits, dim=0)
        val_probs = torch.softmax(all_val_logits_tensor, dim=1)[:, 1].numpy()
        val_auc = roc_auc_score(all_val_labels, val_probs)
        val_ap  = average_precision_score(all_val_labels, val_probs)
        
        print(f"Epoch {epoch+1}/{epochs}")
        print(f"  Train - Loss: {avg_train_loss:.4f} | F1: {train_f1:.4f} | Precision: {train_precision:.4f} | Recall: {train_recall:.4f} | AUC: {train_auc:.4f} | AP: {train_ap:.4f}")
        print(f"  Val   - F1: {val_f1:.4f} | Precision: {val_precision:.4f} | Recall: {val_recall:.4f} | AUC: {val_auc:.4f} | AP: {val_ap:.4f}")
        
        fold_epochs_evolution[f"epoch_{epoch+1}"] = {
            "train_loss": avg_train_loss,
            "train_f1": train_f1,
            "train_precision": train_precision,
            "train_recall": train_recall,
            "train_auc": train_auc,
            "train_ap": train_ap,
            "val_f1": val_f1,
            "val_precision": val_precision,
            "val_recall": val_recall,
            "val_auc": val_auc,
            "val_ap": val_ap,
        }
        
        del all_train_logits, all_logits_tensor, train_probs
        del all_val_logits, all_val_logits_tensor, val_probs
        del all_train_preds, all_train_labels, all_val_preds, all_val_labels
        torch.cuda.empty_cache()
        gc.collect()
    
    cv_results[f"fold_{fold_idx}"] = fold_epochs_evolution
    
    # Cleanup after fold
    del model, optimizer, loss_fn
    del dataset_train, dataset_val, train_dataloader, val_dataloader
    torch.cuda.empty_cache()
    gc.collect()

# Save cross-validation results
json.dump(cv_results, open("cv_results.json", "w"), indent=2)
print(f"\n{'='*80}")
print("Cross-validation complete! Results saved to cv_results.json")
print(f"{'='*80}")



FOLD 1/5


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 4005.83it/s]
DistilBertForSequenceClassification LOAD REPORT from: bert_dist_cache/models--distilbert-base-cased/snapshots/6ea81172465e8b0ad3fddeed32b986cdcdcffcf0
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/home/alant/repos/tal-projet/.venv/lib/python3.12/site-packages/sklearn/metrics/_classificat

Epoch 1/5
  Train - Loss: 0.6984 | F1: 0.4641 | Precision: 0.4847 | Recall: 0.4452 | AUC: 0.4953 | AP: 0.4880
  Val   - F1: 0.0000 | Precision: 0.0000 | Recall: 0.0000 | AUC: 0.4825 | AP: 0.5141


/home/alant/repos/tal-projet/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Epoch 2/5
  Train - Loss: 0.6934 | F1: 0.4929 | Precision: 0.5206 | Recall: 0.4681 | AUC: 0.5305 | AP: 0.5179
  Val   - F1: 0.0000 | Precision: 0.0000 | Recall: 0.0000 | AUC: 0.5081 | AP: 0.5092
Epoch 3/5
  Train - Loss: 0.6917 | F1: 0.5165 | Precision: 0.5312 | Recall: 0.5025 | AUC: 0.5360 | AP: 0.5386
  Val   - F1: 0.6667 | Precision: 0.5000 | Recall: 1.0000 | AUC: 0.5121 | AP: 0.5194
Epoch 4/5
  Train - Loss: 0.6918 | F1: 0.5906 | Precision: 0.5294 | Recall: 0.6679 | AUC: 0.5348 | AP: 0.5447
  Val   - F1: 0.5830 | Precision: 0.5074 | Recall: 0.6850 | AUC: 0.5407 | AP: 0.5599
Epoch 5/5
  Train - Loss: 0.6539 | F1: 0.6047 | Precision: 0.6114 | Recall: 0.5982 | AUC: 0.6584 | AP: 0.6307
  Val   - F1: 0.6516 | Precision: 0.4987 | Recall: 0.9400 | AUC: 0.5138 | AP: 0.5218

FOLD 2/5


...

Epoch 1/5
  Train - Loss: 0.6955 | F1: 0.4610 | Precision: 0.4978 | Recall: 0.4293 | AUC: 0.5116 | AP: 0.4912
  Val   - F1: 0.6667 | Precision: 0.5000 | Recall: 1.0000 | AUC: 0.4691 | AP: 0.4789


/home/alant/repos/tal-projet/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Epoch 2/5
  Train - Loss: 0.6933 | F1: 0.4491 | Precision: 0.4836 | Recall: 0.4191 | AUC: 0.5172 | AP: 0.4988
  Val   - F1: 0.0000 | Precision: 0.0000 | Recall: 0.0000 | AUC: 0.4991 | AP: 0.5047
Epoch 3/5
  Train - Loss: 0.6893 | F1: 0.4599 | Precision: 0.5248 | Recall: 0.4093 | AUC: 0.5434 | AP: 0.5276
  Val   - F1: 0.3587 | Precision: 0.4574 | Recall: 0.2950 | AUC: 0.4622 | AP: 0.4697
Epoch 4/5
  Train - Loss: 0.6472 | F1: 0.5616 | Precision: 0.6220 | Recall: 0.5120 | AUC: 0.6676 | AP: 0.6274
  Val   - F1: 0.5947 | Precision: 0.5017 | Recall: 0.7300 | AUC: 0.5102 | AP: 0.5094
Epoch 5/5
  Train - Loss: 0.5639 | F1: 0.6982 | Precision: 0.6938 | Recall: 0.7027 | AUC: 0.7787 | AP: 0.7587
  Val   - F1: 0.5782 | Precision: 0.5056 | Recall: 0.6750 | AUC: 0.5078 | AP: 0.5045

FOLD 5/5


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 9899.93it/s]
DistilBertForSequenceClassification LOAD REPORT from: bert_dist_cache/models--distilbert-base-cased/snapshots/6ea81172465e8b0ad3fddeed32b986cdcdcffcf0
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/home/alant/repos/tal-projet/.venv/lib/python3.12/site-packages/sklearn/metrics/_classificat

Epoch 1/5
  Train - Loss: 0.6970 | F1: 0.4492 | Precision: 0.4851 | Recall: 0.4183 | AUC: 0.4947 | AP: 0.4982
  Val   - F1: 0.0000 | Precision: 0.0000 | Recall: 0.0000 | AUC: 0.4899 | AP: 0.5001
Epoch 2/5
  Train - Loss: 0.6940 | F1: 0.5081 | Precision: 0.5262 | Recall: 0.4912 | AUC: 0.5245 | AP: 0.5199
  Val   - F1: 0.4740 | Precision: 0.4946 | Recall: 0.4550 | AUC: 0.4947 | AP: 0.5180
Epoch 3/5
  Train - Loss: 0.6906 | F1: 0.5081 | Precision: 0.5206 | Recall: 0.4962 | AUC: 0.5424 | AP: 0.5234
  Val   - F1: 0.2055 | Precision: 0.4906 | Recall: 0.1300 | AUC: 0.4949 | AP: 0.5113
Epoch 4/5
  Train - Loss: 0.6655 | F1: 0.5813 | Precision: 0.5853 | Recall: 0.5773 | AUC: 0.6309 | AP: 0.6219
  Val   - F1: 0.3487 | Precision: 0.5096 | Recall: 0.2650 | AUC: 0.5252 | AP: 0.5255
Epoch 5/5
  Train - Loss: 0.5922 | F1: 0.7017 | Precision: 0.6792 | Recall: 0.7258 | AUC: 0.7468 | AP: 0.7638
  Val   - F1: 0.4311 | Precision: 0.5373 | Recall: 0.3600 | AUC: 0.5218 | AP: 0.5185

Cross-validation complet

## train test normal split 

In [8]:
def load_movies_test(path2data):
    alltxts = []
    c = 0
    with open(path2data, 'r') as file:
        for line in file:        
            alltxts.append(line)
    return alltxts

# load dataset test
alltxt_test = load_movies_test("Test_set/testSentiment.txt")
alltxt_test = np.array(alltxt_test)


In [10]:
# loading the tokenized items

# loading tokenized train and validation
data = torch.load("train_tokens_dist.pt")
input_ids = data["input_ids"]
attention_masks = data["attention_masks"]

data = torch.load("test_tokens_dist.pt")
input_ids_test = data["input_ids"]
attention_masks_test = data["attention_masks"]

# train version

input_ids_train = torch.cat([input_ids, input_ids_test], dim=0)
attention_masks_train = torch.cat([attention_masks, attention_masks_test], dim=0)


In [ ]:
## if not tokenized yet, tokenize test set
"""
from transformers import DistilBertForSequenceClassification, DistilBertTokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-cased') # smaller

# tokenizing test set
maxL = 512 # Max length of the sequence
inputs_tokens_test = []
attention_masks_test = []
for i in range(len(alltxt_test)):
    if(i%2500==0):
        print(i)
    string_tokenized = tokenizer(alltxt_test[i], return_tensors="pt",
                                        add_special_tokens=True,  # add '[CLS]' and '[SEP]'
                                        max_length=maxL,  # set max length
                                        truncation=True,  # truncate longer messages
                                        #pad_to_max_length=True
                                        padding='max_length',  # add padding
                                        return_attention_mask=True)

    inputs_tokens_test.append(string_tokenized['input_ids'])
    attention_masks_test.append(string_tokenized['attention_mask'])

# convert into one big tensor
input_ids_test = torch.cat(inputs_tokens_test, dim=0)
attention_masks_test = torch.cat(attention_masks_test, dim=0)

torch.save({
    "input_ids": input_ids_test,
    "attention_masks": attention_masks_test
}, "pred_test_tokens_dist.pt")
"""


0
2500
5000
7500
10000
12500
15000
17500
20000
22500


In [13]:

# test version

data = torch.load("pred_test_tokens_dist.pt")
input_ids_test = data["input_ids"]
attention_masks_test = data["attention_masks"]


# ------- weights

alllabs = np.where(alllabs == -1, 0, alllabs) # 0-1 labels
val, counts = np.unique(alllabs, return_counts = True)

weights = 1/counts # weights for loss
y_train_tens = torch.tensor(alllabs)


# dataset loading

sample_weights = [weights[label] for label in alllabs]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(alllabs), replacement=True)

dataset_train = TensorDataset(input_ids_train,  attention_masks_train, y_train_tens)
dataset_test = TensorDataset(input_ids_test,  attention_masks_test)

train_dataloader = DataLoader(dataset_train, batch_size=16, sampler=sampler)

test_dataloader = DataLoader(
    dataset_test,
    batch_size=16,
    shuffle=False # f
)


In [17]:

# loading not trained model
from transformers import DistilBertConfig, DistilBertForSequenceClassification

config = DistilBertConfig.from_pretrained('distilbert-base-uncased',
    dropout=0.2,
    attention_dropout=0.2,
    seq_classif_dropout=0.4
)

model = DistilBertForSequenceClassification.from_pretrained("bert_dist_cache/models--distilbert-base-cased/snapshots/6ea81172465e8b0ad3fddeed32b986cdcdcffcf0", config = config,ignore_mismatched_sizes=True)


# Freeze all DistilBERT layers, train only the classifier head
for param in model.distilbert.parameters():
    param.requires_grad = False

# Or unfreeze only the last N encoder layers
for layer in model.distilbert.transformer.layer[-4:]:  # last 4 layers
    for param in layer.parameters():
        param.requires_grad = True


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 8504.44it/s]
DistilBertForSequenceClassification LOAD REPORT from: bert_dist_cache/models--distilbert-base-cased/snapshots/6ea81172465e8b0ad3fddeed32b986cdcdcffcf0
Key                                          | Status     |                                                                                               
---------------------------------------------+------------+-----------------------------------------------------------------------------------------------
vocab_transform.weight                       | UNEXPECTED |                                                                                               
vocab_transform.bias                         | UNEXPECTED |                                                                                               
vocab_projector.bias                         | UNEXPECTED |                                                                                               
vocab

In [ ]:

# pre emtpying cache
gc.collect()
torch.cuda.empty_cache()

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(dev)
print("dev", dev)

class_weights = torch.tensor(weights, dtype=torch.float).to(dev)
loss_fn = CrossEntropyLoss()

# ------ running loop

optimizer = optim.Adam(model.parameters(), lr=2e-5)
epochs = 5 # to test

model.to(dev)
epochs_evolution = {}

# ── Training ─────────────────────────────────────────────────────────────────
for epoch in range(epochs):
    model.train()
    total_loss = 0
    all_train_preds = []
    all_train_labels = []
    all_train_logits = []

    for i, batch in enumerate(train_dataloader):
        print("batch", i)
        b_input_ids, b_attention_mask, b_labels = [t.to(dev) for t in batch]
        model.zero_grad()

        outputs = model(
            input_ids=b_input_ids,
            attention_mask=b_attention_mask,
            # labels=b_labels
        )
        # loss = outputs.loss
        logits = outputs.logits
        loss = loss_fn(logits, b_labels)

        total_loss += loss.item()
        loss.backward()
        optimizer.step()

        preds = torch.argmax(logits, dim=1)
        all_train_preds.extend(preds.cpu().numpy())
        all_train_labels.extend(b_labels.cpu().numpy())
        all_train_logits.append(logits.detach().cpu())

        del b_input_ids, b_attention_mask, b_labels
        del outputs, loss, logits, preds
        torch.cuda.empty_cache()
        gc.collect()

    avg_train_loss = total_loss / len(train_dataloader)

    # Training metrics
    train_f1        = f1_score(all_train_labels, all_train_preds, average="macro")
    train_precision = precision_score(all_train_labels, all_train_preds, average="macro")
    train_recall    = recall_score(all_train_labels, all_train_preds, average="macro")

    all_logits_tensor = torch.cat(all_train_logits, dim=0)
    train_probs = torch.softmax(all_logits_tensor, dim=1)[:, 1].numpy()
    train_auc = roc_auc_score(all_train_labels, train_probs)
    train_ap  = average_precision_score(all_train_labels, train_probs)

    print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_train_loss:.4f} | "
          f"F1: {train_f1:.4f} | Precision: {train_precision:.4f} | Recall: {train_recall:.4f} | "
          f"AUC: {train_auc:.4f} | AP: {train_ap:.4f}")

    epochs_evolution[f"epoch_{epoch+1}"] = {   # fixed key to track each epoch
        "loss": avg_train_loss,
        "f1": train_f1,
        "precision": train_precision,
        "recall": train_recall,
        "auc": train_auc,
        "ap": train_ap
    }

    del all_train_logits, all_logits_tensor, train_probs
    del all_train_preds, all_train_labels
    torch.cuda.empty_cache()
    gc.collect()

json.dump(epochs_evolution, open("train_res.json", "w"))
print("Training metrics saved to train_res.json")

# ── Prediction on unlabeled test set ─────────────────────────────────────────
model.eval()
all_preds = []
all_probs = []

with torch.no_grad():
    for batch in test_dataloader:
        b_input_ids, b_attention_mask = [t.to(dev) for t in batch]  # no labels

        outputs = model(input_ids=b_input_ids, attention_mask=b_attention_mask)
        logits  = outputs.logits
        probs   = torch.softmax(logits, dim=1)[:, 1]
        preds   = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

        del b_input_ids, b_attention_mask
        del outputs, logits, probs, preds
        torch.cuda.empty_cache()
        gc.collect()

# Save predictions and probabilities
np.save("PRED_test_preds.npy", np.array(all_preds))
np.save("PRED_test_probs.npy", np.array(all_probs))
print(f"Predictions saved — {len(all_preds)} samples")
print(f"Class distribution: { {v: int((np.array(all_preds)==v).sum()) for v in np.unique(all_preds)} }")

print("predictions unique", np.unique(all_preds, return_counts=True))

# ── Final cleanup ─────────────────────────────────────────────────────────────
del all_preds, all_probs
torch.cuda.empty_cache()
gc.collect()

dev cuda
batch 0
batch 1
batch 2
batch 3
batch 4
batch 5
batch 6
batch 7
batch 8
batch 9
batch 10
batch 11
batch 12
batch 13
batch 14
batch 15
batch 16
batch 17
batch 18
batch 19
batch 20
batch 21
batch 22
batch 23
batch 24
batch 25
batch 26
batch 27
batch 28
batch 29
batch 30
batch 31
batch 32
batch 33
batch 34
batch 35
batch 36
batch 37
batch 38
batch 39
batch 40
batch 41
batch 42
batch 43
batch 44
batch 45
batch 46
batch 47
batch 48
batch 49
batch 50
batch 51
batch 52
batch 53
batch 54
batch 55
batch 56
batch 57
batch 58
batch 59
batch 60
batch 61
batch 62
batch 63
batch 64
batch 65
batch 66
batch 67
batch 68
batch 69
batch 70
batch 71
batch 72
batch 73
batch 74
batch 75
batch 76
batch 77
batch 78
batch 79
batch 80
batch 81
batch 82
batch 83
batch 84
batch 85
batch 86
batch 87
batch 88
batch 89
batch 90
batch 91
batch 92
batch 93
batch 94
batch 95
batch 96
batch 97
batch 98
batch 99
batch 100
batch 101
batch 102
batch 103
batch 104
batch 105
batch 106
batch 107
batch 108
batch 109
b

0